In [1]:
!pip install git+https://github.com/facebookresearch/sam2.git

  Cloning https://github.com/facebookresearch/sam2.git to /tmp/pip-req-build-h479nb7t
  Running command git clone --filter=blob:none --quiet https://github.com/facebookresearch/sam2.git /tmp/pip-req-build-h479nb7t
  Resolved https://github.com/facebookresearch/sam2.git to commit 2b90b9f5ceec907a1c18123530e92e794ad901a4
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.2/42.2 kB 1.6 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 154.5/154.5 kB 6.2 MB/s eta 0:00:00
  Created wheel for SAM-2: filename=sam_2-1.0-cp312-cp312-linux_x86_64.whl size=504949 sha256=740a721eca109c147337083e7d58524c548c6f4551891f82a7df8de6e86b5830
  Stored in directory: /tmp/pip-ephem-wheel-cache-o_s26wjy/wheels/25/a3/8a/abd69dc6a6926b5e75c24810afac36c7b49b5c0f8a100147d6
  Created wheel for iopath: filename=iopath-0.1.10-py3-non

In [2]:
!wget -q https://dl.fbaipublicfiles.com/segment_anything_2/072824/sam2_hiera_small.pt

In [ ]:
import os
import cv2
import torch
import numpy as np
import shutil
from tqdm import tqdm
from PIL import Image

from transformers import AutoProcessor, AutoModelForZeroShotObjectDetection
from sam2.build_sam import build_sam2_video_predictor

# ================= CONFIGURATION =================
INPUT_VIDEOS = [
    "/kaggle/input/datasets/gonoszgonosz/rat-test-video/test.mp4",
    "/kaggle/input/datasets/gonoszgonosz/rat-test-video/test2.mp4",
    "/kaggle/input/datasets/gonoszgonosz/rat-test-video/test3.mp4",
]
OUTPUT_VIDEOS = [
    "/kaggle/working/Grounded_SAM2_Video_Output_1.mp4",
    "/kaggle/working/Grounded_SAM2_Video_Output_2.mp4",
    "/kaggle/working/Grounded_SAM2_Video_Output_3.mp4",
]
TEMP_FRAME_DIR = "/kaggle/working/sam2_temp_frames"

TEXT_PROMPT = "rat."
BOX_THRESHOLD = 0.35
TEXT_THRESHOLD = 0.25

SAM2_CHECKPOINT = "sam2_hiera_small.pt"
MODEL_CFG = "sam2_hiera_s.yaml"

device = "cuda" if torch.cuda.is_available() else "cpu"
# =================================================

def extract_frames(video_path, output_dir):
    if os.path.exists(output_dir):
        shutil.rmtree(output_dir)
    os.makedirs(output_dir)

    cap = cv2.VideoCapture(video_path)
    fps = cap.get(cv2.CAP_PROP_FPS)
    w = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    h = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

    frame_idx = 0
    while True:
        ret, frame = cap.read()
        if not ret: break
        cv2.imwrite(os.path.join(output_dir, f"{frame_idx:05d}.jpg"), frame)
        frame_idx += 1

    cap.release()
    return fps, w, h, frame_idx

def apply_overlay(image, mask, color=(0, 255, 0), alpha=0.5):
    overlay = np.full_like(image, color)
    blended = cv2.addWeighted(image, 1 - alpha, overlay, alpha, 0)
    res = image.copy()
    res[mask > 0] = blended[mask > 0]
    return res

def process_video(input_video, output_video, predictor, gd_model, gd_processor):
    print(f"\n--- PROCESSING: {input_video} ---")

    print("--- EXTRACTING FRAMES ---")
    fps, w, h, total_frames = extract_frames(input_video, TEMP_FRAME_DIR)
    print(f"Extracted {total_frames} frames")

    print("--- INITIALIZING SAM 2 MEMORY BANK ---")
    inference_state = predictor.init_state(
        video_path=TEMP_FRAME_DIR,
        offload_video_to_cpu=True,
        offload_state_to_cpu=True
    )

    print("--- GENERATING FRAME 0 PROMPT VIA GROUNDING DINO ---")
    first_frame_path = os.path.join(TEMP_FRAME_DIR, "00000.jpg")
    first_frame_img = cv2.imread(first_frame_path)
    first_frame_rgb = cv2.cvtColor(first_frame_img, cv2.COLOR_BGR2RGB)
    pil_image = Image.fromarray(first_frame_rgb)

    inputs = gd_processor(images=pil_image, text=TEXT_PROMPT, return_tensors="pt").to(device)
    with torch.no_grad():
        outputs = gd_model(**inputs)

    results = gd_processor.post_process_grounded_object_detection(
        outputs, inputs.input_ids, threshold=BOX_THRESHOLD, text_threshold=TEXT_THRESHOLD,
        target_sizes=[pil_image.size[::-1]]
    )[0]

    if len(results["boxes"]) == 0:
        print("CRITICAL ERROR: Grounding DINO could not find the rat in Frame 0. Skipping video.")
        predictor.reset_state(inference_state)
        shutil.rmtree(TEMP_FRAME_DIR)
        return

    best_box_idx = torch.argmax(results["scores"]).item()
    init_box = results["boxes"][best_box_idx].cpu().numpy()
    print(f"Bounding Box Found: {init_box}")

    _, out_obj_ids, out_mask_logits = predictor.add_new_points_or_box(
        inference_state=inference_state, frame_idx=0, obj_id=1, box=init_box
    )

    print("--- PROPAGATING THROUGH VIDEO ---")
    fourcc = cv2.VideoWriter_fourcc(*'mp4v')
    out_video = cv2.VideoWriter(output_video, fourcc, fps, (w, h))

    for out_frame_idx, out_obj_ids, out_mask_logits in predictor.propagate_in_video(inference_state):
        frame_path = os.path.join(TEMP_FRAME_DIR, f"{out_frame_idx:05d}.jpg")
        frame = cv2.imread(frame_path)
        mask = (out_mask_logits[0] > 0.0).cpu().numpy().squeeze().astype(np.uint8)

        res_frame = apply_overlay(frame, mask)
        cv2.putText(res_frame, "Grounded-SAM 2 Temporal Tracking", (50, 50), cv2.FONT_HERSHEY_SIMPLEX, 1, (255, 255, 255), 2)
        if out_frame_idx == 0:
            x1, y1, x2, y2 = map(int, init_box)
            cv2.rectangle(res_frame, (x1, y1), (x2, y2), (0, 0, 255), 2)
            cv2.putText(res_frame, "DINO Prompt Box", (x1, y1 - 10), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 0, 255), 2)

        out_video.write(res_frame)

    out_video.release()
    predictor.reset_state(inference_state)
    shutil.rmtree(TEMP_FRAME_DIR)
    print(f"--- DONE. SAVED TO {output_video} ---")

def main():
    print("--- LOADING GROUNDING DINO ---")
    gd_processor = AutoProcessor.from_pretrained("IDEA-Research/grounding-dino-base")
    gd_model = AutoModelForZeroShotObjectDetection.from_pretrained("IDEA-Research/grounding-dino-base").to(device)
    gd_model.eval()

    print("--- LOADING SAM 2 VIDEO PREDICTOR ---")
    predictor = build_sam2_video_predictor(MODEL_CFG, SAM2_CHECKPOINT, device=device)

    for input_video, output_video in zip(INPUT_VIDEOS, OUTPUT_VIDEOS):
        process_video(input_video, output_video, predictor, gd_model, gd_processor)

if __name__ == "__main__":
    main()
